# Librerias

In [1]:
import pandas as pd
import glob # Sirve para buscar varios csvs
import os # Permite que python interactue con OS (Usado unicamente para path.join)
from pathlib import Path

In [1]:
from ydata_profiling import ProfileReport
print("✓ ydata_profiling imported successfully!")

c:\Users\birob\OneDrive\Desktop\ProyectosVScode\COMCAIIAM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ ydata_profiling imported successfully!


# Read y Concat de csvs

## Benign

In [2]:
# Ruta de la carpeta para archivos en Windows local
ruta_carpeta_b = r'C:\Users\birob\OneDrive\Desktop\ProyectosVScode\COMCAIIAM\data\benign_data'

# Obtener todos los benignos usando la ruta completa
archivos_benignos = glob.glob(os.path.join(ruta_carpeta_b, "benign_samples_*.csv"))

# Cargarlos todos en una lista
lista_df = [pd.read_csv(f) for f in archivos_benignos]
df_benignos = pd.concat(lista_df, ignore_index=True)

## Attacks

In [3]:
# Mismo proceso que con los benignos

ruta_carpeta_a = r'C:\Users\birob\OneDrive\Desktop\ProyectosVScode\COMCAIIAM\data\attack_data'

archivos_ataques = glob.glob(os.path.join(ruta_carpeta_a, "attack_samples_*.csv"))
lista_ataques = []

# Se utiliza un print debido al mayor tiempo de espera de carga en los archivos
#
for f in archivos_ataques:
    print(f"Leyendo {os.path.basename(f)}...")
    temp_df = pd.read_csv(f)
    lista_ataques.append(temp_df)

df_ataques = pd.concat(lista_ataques, ignore_index=True)

Leyendo attack_samples_10sec.csv...
Leyendo attack_samples_1sec.csv...
Leyendo attack_samples_2sec.csv...
Leyendo attack_samples_3sec.csv...
Leyendo attack_samples_4sec.csv...
Leyendo attack_samples_5sec.csv...
Leyendo attack_samples_6sec.csv...
Leyendo attack_samples_7sec.csv...
Leyendo attack_samples_8sec.csv...
Leyendo attack_samples_9sec.csv...


# Analisis del dataset

## Conteo de Clases

In [ ]:
n_benignos = len(df_benignos)
n_ataques = len(df_ataques)

print(f"Total Benignos: {n_benignos}")
print(f"Total Ataques: {n_ataques}")


Total Benignos: 400672
Total Ataques: 284999


In [ ]:
n_categoriasAtaques = df_ataques['label2'].value_counts()

print(len(n_categoriasAtaques))
print(n_categoriasAtaques)

7
label2
recon         105848
dos            57736
ddos           56692
mitm           25490
malware        24177
web             9040
bruteforce      6016
Name: count, dtype: int64


In [ ]:
n_tiposAtaques = df_ataques['label3'].value_counts()

print(len(n_tiposAtaques)) # Paper menciona 50
print(n_tiposAtaques)

## Features ##

In [ ]:
print(df_benignos.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400672 entries, 0 to 400671
Data columns (total 94 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   device_name                           400672 non-null  object 
 1   device_mac                            400672 non-null  object 
 2   label_full                            400672 non-null  object 
 3   label1                                400672 non-null  object 
 4   label2                                400672 non-null  object 
 5   label3                                400672 non-null  object 
 6   label4                                400672 non-null  object 
 7   timestamp                             400672 non-null  object 
 8   timestamp_start                       400672 non-null  object 
 9   timestamp_end                         400672 non-null  object 
 10  log_data-ranges_avg                   400672 non-null  float64
 11  

In [ ]:
print(df_ataques.info())

## Possible views ##

Context view:

- Identifiers: device_name, device_mac, network_ips_src, network_ips_dst, network_macs_src, network_macs_dst, network_ips_all, network_macs_all.
- Temporal: timestamp, timestamp_start, timestamp_end.

NetworkFlow view:

- Packet Dynamics: network_packet-size_*, network_ip-length_*, network_payload-length_*, network_header-length_*, network_mss_*.
- Traffic Volume: network_packets_all_count, network_packets_dst_count, network_packets_src_count, network_interval-packets.
- Connection Info: network_ports_src, network_ports_dst, network_protocols_all, network_ttl_*, network_window-size_*.
- TCP/IP Control (TCP Flags): Todas las de network_tcp-flags-* (ACK, FIN, PSH, RST, SYN, URG) e network_ip-flags_*.
- Fragmentation: network_fragmentation-score, network_fragmented-packets.

Log view

- Log Behavior: log_data-ranges_*, log_data-types, log_data-types_count.
- Log Intensity: log_messages_count, log_interval-messages.

# Sample

In [4]:
df_unificado = pd.concat([df_benignos, df_ataques], ignore_index=True)
print(f"Filas totales: {len(df_unificado)}")

Filas totales: 685671


In [5]:
# Supongamos que quieres 8,000 registros en total
n_test = 4000 

df_b_sample = df_unificado[df_unificado['label2'] == 'benign'].sample(n=n_test, random_state=42)
df_a_sample = df_unificado[df_unificado['label2'] != 'benign'].sample(n=n_test, random_state=42)

df_pruebas = pd.concat([df_b_sample, df_a_sample], ignore_index=True).sample(frac=1)

print(f"Muestra balanceada: {len(df_pruebas)} registros")

Muestra balanceada: 8000 registros


In [6]:
n_categoriasAtaques = df_pruebas['label2'].value_counts()

print(len(n_categoriasAtaques))
print(n_categoriasAtaques)

8
label2
benign        4000
recon         1555
ddos           794
dos            780
malware        335
mitm           326
web            127
bruteforce      83
Name: count, dtype: int64


In [ ]:
columnas_a_quitar = ['label_full', 'label1', 'label3', 'label4']
df_pruebas = df_pruebas.drop(columns=columnas_a_quitar)

df_pruebas = df_pruebas.rename(columns={'label2': 'class'})

cols = list(df_pruebas.columns)
cols.insert(0, cols.pop(cols.index('class')))
df_pruebas = df_pruebas[cols]

df_pruebas.info()

In [8]:
import re

def renombrar_por_vistas(df):
    nuevas_columnas = {}
    
    for col in df.columns:
        # Saltamos la columna 'class' para que no cambie
        if col == 'class':
            nuevas_columnas[col] = col
            continue
            
        # Lógica para V1: Context View
        if any(x in col for x in ['device_', 'ips_', 'macs_', 'timestamp']):
            nuevas_columnas[col] = f"v1_{col}"
            
        # Lógica para V3: Log View
        elif 'log_' in col:
            nuevas_columnas[col] = f"v3_{col}"
            
        # Lógica para V2: NetworkFlow View (Todo lo que empieza con 'network_')
        # Excluimos las que ya atrapo V1 (como ips o macs)
        elif 'network_' in col:
            nuevas_columnas[col] = f"v2_{col}"
            
        else:
            # Por si queda alguna suelta
            nuevas_columnas[col] = col
            
    return df.rename(columns=nuevas_columnas)

df_pruebas = renombrar_por_vistas(df_pruebas)

print(df_pruebas.columns.tolist())

['class', 'v1_device_name', 'v1_device_mac', 'v1_timestamp', 'v1_timestamp_start', 'v1_timestamp_end', 'v3_log_data-ranges_avg', 'v3_log_data-ranges_max', 'v3_log_data-ranges_min', 'v3_log_data-ranges_std_deviation', 'v3_log_data-types', 'v3_log_data-types_count', 'v3_log_interval-messages', 'v3_log_messages_count', 'v2_network_fragmentation-score', 'v2_network_fragmented-packets', 'v2_network_header-length_avg', 'v2_network_header-length_max', 'v2_network_header-length_min', 'v2_network_header-length_std_deviation', 'v2_network_interval-packets', 'v2_network_ip-flags_avg', 'v2_network_ip-flags_max', 'v2_network_ip-flags_min', 'v2_network_ip-flags_std_deviation', 'v2_network_ip-length_avg', 'v2_network_ip-length_max', 'v2_network_ip-length_min', 'v2_network_ip-length_std_deviation', 'v1_network_ips_all', 'v1_network_ips_all_count', 'v1_network_ips_dst', 'v1_network_ips_dst_count', 'v1_network_ips_src', 'v1_network_ips_src_count', 'v1_network_macs_all', 'v1_network_macs_all_count', 'v1_

In [9]:
print(f"Columnas: {len(df_pruebas.columns)}")

df_numerico = df_pruebas.select_dtypes(include=['number'])

if 'class' in df_pruebas.columns:
    df_pruebas = pd.concat([df_pruebas[['class']], df_numerico], axis=1)
else:
    df_pruebas = df_numerico

print(f"Columnas restantes: {len(df_pruebas.columns)}")
print(df_pruebas.dtypes.value_counts())

Columnas: 90
Columnas restantes: 72
float64    47
int64      24
object      1
Name: count, dtype: int64


In [10]:
ruta_salida = Path.cwd().parent / "data" / "htad" / "test_10k.csv"

# Guardamos
df_pruebas.to_csv(ruta_salida, index=False)

print(f"Archivo guardado con éxito en: {ruta_salida}")

Archivo guardado con éxito en: c:\Users\birob\OneDrive\Desktop\ProyectosVScode\COMCAIIAM\data\htad\test_10k.csv


# Balanceo

## Posible solucion de balanceo

In [ ]:
# To do